In [10]:
import pandas as pd
import numpy as np

In [11]:
df_workers = pd.read_csv("worker_master_dataset.csv")

In [12]:
df_workers.head()

,worker_id,name,skill,experience,rating,workload,availability,latitude,longitude
0,W0001,Arun Mishra,Painting,17,3.7,8,1,28.508227,77.272012
1,W0002,Arun Gupta,Carpentry,6,3.4,1,1,28.613568,77.107222
2,W0003,Mohit Kumar,Gardening,2,3.8,7,1,28.701854,77.306128
3,W0004,Mohit Verma,Carpentry,12,4.6,6,1,28.496527,77.609808
4,W0005,Vikas Joshi,Electrical,6,4.3,3,1,28.829180,77.532576


In [13]:
df_workers.shape

(1000, 9)

In [14]:
np.random.seed(42)

In [15]:
services = {
    "Plumbing": [
        "Pipe Leakage",
        "Tap Repair",
        "Bathroom Repair",
        "Water Tank"
    ],
    
    "Electrical": [
        "Fan Repair",
        "Wiring",
        "Switch Repair",
        "Appliance Installation"
    ],
    
    "Carpentry": [
        "Furniture Repair",
        "Door Repair",
        "Cabinet Repair",
        "Woodwork"
    ],
    
    "Cleaning": [
        "House Cleaning",
        "Deep Cleaning",
        "Bathroom Cleaning",
        "Kitchen Cleaning"
    ],
    
    "Painting": [
        "Wall Painting",
        "Room Painting",
        "Furniture Painting",
        "Exterior Painting"
    ],
    
    "Gardening": [
        "Lawn Maintenance",
        "Plant Care",
        "Garden Cleaning",
        "Tree Trimming"
    ]
}

In [16]:
n_requests = 2000

In [17]:
request_id = [
    f"R{i:04d}" for i in range(1, n_requests + 1)
]

In [18]:
requested_service = np.random.choice(
    list(services.keys()),
    size=n_requests
)

In [19]:
requested_problem = [
    np.random.choice(services[service])
    for service in requested_service
]

In [20]:
customer_latitude = np.round(
    np.random.uniform(28.40, 29.20, size=n_requests),
    6
)

customer_longitude = np.round(
    np.random.uniform(76.80, 77.80, size=n_requests),
    6
)

In [21]:
candidate_rows = []

for i in range(n_requests):
    
    service = requested_service[i]
    
    matching_workers = df_workers[
        df_workers["skill"] == service
    ]
    
    non_matching_workers = df_workers[
        df_workers["skill"] != service
    ]
    
    # 7 workers with the required skill
    selected_matching = matching_workers.sample(
        n=7,
        random_state=i
    )
    
    # 3 workers with other skills
    selected_non_matching = non_matching_workers.sample(
        n=3,
        random_state=i
    )
    
    selected_workers = pd.concat([
        selected_matching,
        selected_non_matching
    ])
    
    for _, worker in selected_workers.iterrows():
        candidate_rows.append({
            "request_id": request_id[i],
            "requested_service": service,
            "requested_problem": requested_problem[i],
            "customer_latitude": customer_latitude[i],
            "customer_longitude": customer_longitude[i],
            "worker_id": worker["worker_id"],
            "worker_name": worker["name"],
            "worker_skill": worker["skill"],
            "worker_experience": worker["experience"],
            "worker_rating": worker["rating"],
            "worker_workload": worker["workload"],
            "worker_availability": worker["availability"],
            "worker_latitude": worker["latitude"],
            "worker_longitude": worker["longitude"]
        })

In [22]:
matching_df = pd.DataFrame(candidate_rows)

In [23]:
matching_df.head()

,request_id,requested_service,requested_problem,customer_latitude,customer_longitude,worker_id,worker_name,worker_skill,worker_experience,worker_rating,worker_workload,worker_availability,worker_latitude,worker_longitude
0,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0918,Sunil Yadav,Cleaning,2,3.3,3,1,28.580210,76.899841
1,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0387,Ravi Agarwal,Cleaning,1,3.2,3,1,29.131972,77.779809
2,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0799,Pankaj Gupta,Cleaning,14,4.8,6,1,29.174896,77.646124
3,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0404,Vikas Sharma,Cleaning,6,3.2,7,1,28.889871,77.045030
4,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0674,Arun Gupta,Cleaning,8,3.5,2,1,28.553709,76.811313


In [24]:
matching_df.shape

(20000, 14)

In [25]:
matching_df["requested_service"].value_counts()

requested_service
Plumbing      3520
Painting      3370
Gardening     3300
Electrical    3290
Cleaning      3260
Carpentry     3260
Name: count, dtype: int64

In [26]:
matching_df["worker_skill"].value_counts()

worker_skill
Painting      3397
Plumbing      3395
Gardening     3356
Cleaning      3334
Carpentry     3289
Electrical    3229
Name: count, dtype: int64

In [27]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [28]:
matching_df["distance_km"] = haversine_distance(
    matching_df["customer_latitude"],
    matching_df["customer_longitude"],
    matching_df["worker_latitude"],
    matching_df["worker_longitude"]
)

matching_df["distance_km"] = matching_df["distance_km"].round(2)

In [29]:
matching_df[[
    "customer_latitude",
    "customer_longitude",
    "worker_latitude",
    "worker_longitude",
    "distance_km"
]].head()

,customer_latitude,customer_longitude,worker_latitude,worker_longitude,distance_km
0,28.637382,76.949141,28.580210,76.899841,7.97
1,28.637382,76.949141,29.131972,77.779809,97.80
2,28.637382,76.949141,29.174896,77.646124,90.42
3,28.637382,76.949141,28.889871,77.045030,29.59
4,28.637382,76.949141,28.553709,76.811313,16.36


In [30]:
matching_df["skill_match"] = (
    matching_df["requested_service"] ==
    matching_df["worker_skill"]
).astype(int)

In [31]:
matching_df[[
    "requested_service",
    "worker_skill",
    "skill_match",
    "distance_km"
]].head(10)

,requested_service,worker_skill,skill_match,distance_km
0,Cleaning,Cleaning,1,7.97
1,Cleaning,Cleaning,1,97.80
2,Cleaning,Cleaning,1,90.42
3,Cleaning,Cleaning,1,29.59
4,Cleaning,Cleaning,1,16.36
5,Cleaning,Cleaning,1,24.55
6,Cleaning,Cleaning,1,4.83
7,Cleaning,Gardening,0,64.38
8,Cleaning,Plumbing,0,51.52
9,Cleaning,Carpentry,0,78.34


In [32]:
matching_df["distance_score"] = 1 / (1 + matching_df["distance_km"])

In [33]:
matching_df.to_csv(
    "matching_training_dataset.csv",
    index=False
)

In [34]:
matching_df["experience_score"] = (
    matching_df["worker_experience"] / 20
)

matching_df["rating_score"] = (
    matching_df["worker_rating"] / 5
)

matching_df["availability_score"] = (
    matching_df["worker_availability"]
)

matching_df["workload_score"] = (
    1 - matching_df["worker_workload"] / 10
)

In [35]:
matching_df["distance_score"] = (
    1 / (1 + matching_df["distance_km"])
)

In [36]:
match_probability = (
    0.35 * matching_df["skill_match"] +
    0.20 * matching_df["distance_score"] +
    0.15 * matching_df["availability_score"] +
    0.10 * matching_df["experience_score"] +
    0.10 * matching_df["rating_score"] +
    0.10 * matching_df["workload_score"]
)

In [37]:
match_probability = np.clip(
    match_probability,
    0.05,
    0.95
)

In [38]:
matching_df["successful_match"] = np.random.binomial(
    1,
    match_probability
)

In [39]:
matching_df["successful_match"].value_counts()

successful_match
1    11134
0     8866
Name: count, dtype: int64

In [40]:
matching_df.to_csv(
    "matching_training_dataset.csv",
    index=False
)

In [41]:
matching_df.shape

(20000, 22)

In [42]:
matching_df.info()
matching_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   request_id           20000 non-null  str    
 1   requested_service    20000 non-null  str    
 2   requested_problem    20000 non-null  str    
 3   customer_latitude    20000 non-null  float64
 4   customer_longitude   20000 non-null  float64
 5   worker_id            20000 non-null  str    
 6   worker_name          20000 non-null  str    
 7   worker_skill         20000 non-null  str    
 8   worker_experience    20000 non-null  int64  
 9   worker_rating        20000 non-null  float64
 10  worker_workload      20000 non-null  int64  
 11  worker_availability  20000 non-null  int64  
 12  worker_latitude      20000 non-null  float64
 13  worker_longitude     20000 non-null  float64
 14  distance_km          20000 non-null  float64
 15  skill_match          20000 non-null  int64  
 1

,request_id,requested_service,requested_problem,customer_latitude,customer_longitude,worker_id,worker_name,worker_skill,worker_experience,worker_rating,...,worker_latitude,worker_longitude,distance_km,skill_match,distance_score,experience_score,rating_score,availability_score,workload_score,successful_match
0,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0918,Sunil Yadav,Cleaning,2,3.3,...,28.580210,76.899841,7.97,1,0.111483,0.10,0.66,1,0.7,1
1,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0387,Ravi Agarwal,Cleaning,1,3.2,...,29.131972,77.779809,97.80,1,0.010121,0.05,0.64,1,0.7,1
2,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0799,Pankaj Gupta,Cleaning,14,4.8,...,29.174896,77.646124,90.42,1,0.010939,0.70,0.96,1,0.4,1
3,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0404,Vikas Sharma,Cleaning,6,3.2,...,28.889871,77.045030,29.59,1,0.032690,0.30,0.64,1,0.3,1
4,R0001,Cleaning,Bathroom Cleaning,28.637382,76.949141,W0674,Arun Gupta,Cleaning,8,3.5,...,28.553709,76.811313,16.36,1,0.057604,0.40,0.70,1,0.8,1


In [43]:
matching_df["successful_match"].value_counts(normalize=True)

successful_match
1    0.5567
0    0.4433
Name: proportion, dtype: float64

In [45]:
matching_df[[
    "requested_service",
    "requested_problem",
    "worker_skill",
    "worker_experience",
    "worker_rating",
    "worker_workload",
    "worker_availability",
    "distance_km",
    "skill_match",
    "successful_match"
]].head(10)

,requested_service,requested_problem,worker_skill,worker_experience,worker_rating,worker_workload,worker_availability,distance_km,skill_match,successful_match
0,Cleaning,Bathroom Cleaning,Cleaning,2,3.3,3,1,7.97,1,1
1,Cleaning,Bathroom Cleaning,Cleaning,1,3.2,3,1,97.80,1,1
2,Cleaning,Bathroom Cleaning,Cleaning,14,4.8,6,1,90.42,1,1
3,Cleaning,Bathroom Cleaning,Cleaning,6,3.2,7,1,29.59,1,1
4,Cleaning,Bathroom Cleaning,Cleaning,8,3.5,2,1,16.36,1,1
5,Cleaning,Bathroom Cleaning,Cleaning,9,4.1,1,1,24.55,1,0
6,Cleaning,Bathroom Cleaning,Cleaning,4,4.7,8,1,4.83,1,1
7,Cleaning,Bathroom Cleaning,Gardening,10,4.5,6,1,64.38,0,1
8,Cleaning,Bathroom Cleaning,Plumbing,14,4.3,10,1,51.52,0,0
9,Cleaning,Bathroom Cleaning,Carpentry,7,4.4,6,1,78.34,0,0
